# A Reproducible Multimodal Framework for Glacial Lake Outburst Flood (GLOF) Susceptibility and Monitoring

**Interview project | Computer Vision & Pattern Recognition Unit, Indian Statistical Institute, Kolkata**  
**Prepared:** 29 October 2026 · **Interview date in the notice:** Friday, 30 October 2026  
**Primary study region:** Kashmir Himalaya (open lake-level susceptibility dataset)  
**Satellite case study:** South Lhonak Lake, Sikkim, India (October 2023 event)

> **Scientific scope statement.** This notebook is a reproducible research prototype for *susceptibility screening and monitoring*. It does **not** claim to predict the exact time of a future outburst. A lake-susceptibility label, a historical event inventory, a satellite change detector, and a calibrated time-to-event forecast are different objects. The notebook keeps them separate.

## Executive summary

Glacial lake outburst floods are compound hazards: a trigger (for example, an ice/rock avalanche, intense melt or rainfall, or seismic shaking) can generate displacement waves, overtop or destabilise a dam, release water and sediment, and propagate a flood downstream. A useful operational system therefore needs more than a single image classifier.

This project combines:

1. **Earth-science susceptibility factors** — lake area and growth, glacier contact/type, dam material, slope, lake cascades, permafrost and seismic context.
2. **Statistical change analysis** — lake-area change where paired historical measurements exist, uncertainty intervals, and non-parametric tests.
3. **Machine learning** — transparent Logistic Regression and Random Forest baselines with spatially grouped cross-validation, evaluated against the *published susceptibility classes*.
4. **Satellite monitoring** — an optional live Sentinel-2 L2A extraction around South Lhonak Lake using Green/NIR NDWI and scene-classification masking, explicitly labelled exploratory.
5. **Event-inventory analysis** — temporal summaries from the public High Mountain Asia GLOF database where download access is available.
6. **Operational interpretation** — an alerting architecture linking hazard susceptibility, changing precursors, and downstream exposure.

### What counts as a valid result?

- A high AUC on a class derived from AHP factors is **not** proof that a future GLOF is predicted. It only measures how well a model reproduces that susceptibility classification.
- A historical event inventory contains events, but generally not all non-event lake-years. It cannot alone supply valid negative examples for ordinary supervised prediction.
- Spatially adjacent lakes are correlated. Random row-wise splitting can exaggerate performance, so the ML benchmark uses groups based on spatial clusters.
- True forecasting needs a harmonised **lake × time** dataset, pre-event predictors, reliable event dates, comparable non-event periods, and a time- and geography-held-out evaluation.

---

## 1. Research question and prediction targets

We separate three tasks:

| Task | Output | Data and validation | Role in this prototype |
|---|---|---|---|
| **Susceptibility screening** | Relative ranking / low-to-very-high class per lake | Lake characteristics and expert/geophysical risk labels; compare ranks and classes | Main quantitative ML benchmark |
| **Change detection / monitoring** | Change in water-like surface or lake area between dates | Multitemporal satellite scenes; cloud/shadow/snow QA and boundary checks | Optional South Lhonak demonstration + lake-dynamics statistics |
| **Outburst forecasting** | Probability of outburst within a stated horizon, e.g. 1 year | Lake-year panel with positives and carefully sampled negatives; temporal/spatial external validation | **Research goal, not claimed as solved here** |

For downstream risk, susceptibility should be combined with exposure and vulnerability. A lake that is likely to fail but drains into an uninhabited valley has a different *risk* from a similarly susceptible lake above a town or hydropower plant.

## 2. State of the art: what is useful and implementable?

| Method family | What it does well | Important limitations | How this project uses it |
|---|---|---|---|
| **GIS + expert scoring / AHP / multi-criteria decision analysis** | Transparent integration of geomorphology, dam material, lake morphology, cascade connectivity, permafrost and seismicity; practical where labels are scarce | Weights depend on expert assumptions and regional setting; susceptibility is not a calibrated probability of failure | Use a published lake susceptibility inventory; treat its classes as a benchmark, not truth about future outcomes |
| **Optical remote sensing (Landsat / Sentinel-2), NDWI/MNDWI and manual delineation** | Repeated lake outlines and area-change signals; accessible open imagery | Clouds, topographic shadows, snow/ice, sediment, seasonal variation and threshold choice create errors | Optional Sentinel-2 pre/post demonstration; no claim of a validated lake outline without comparison to authoritative boundaries |
| **SAR (Sentinel-1) for cloudy scenes** | Radar can monitor through cloud and at night; useful in monsoon/high-mountain conditions | Speckle, layover/shadow, complex terrain and specialised preprocessing | Recommended operational input, especially alongside optical observations |
| **CNN segmentation (U-Net, DeepLab family) / semantic segmentation** | Pixel-level mapping can outperform hand thresholds with adequate annotated imagery | Label cost, domain shift across regions/sensors/seasons, and limited compute; a segmentation metric is not an outburst-prediction metric | Proposed next step after building a georeferenced multi-date annotation set; deliberately not faked without labelled masks |
| **Tabular ML (Logistic Regression, Random Forest, gradient boosting / XGBoost)** | Handles mixed variables and non-linear interactions; feature importance can guide field review | Rare events, confounding, spatial leakage, label noise and poor transferability | Use simple baselines to emulate the provided susceptibility classes; report spatial CV and its limits |
| **Time-series statistics (Mann–Kendall / Sen slope, change points, anomaly detection)** | Distinguishes persistent growth or deformation from seasonality/noise | Sparse scene cadence and inconsistent cloud-free observations can destabilise trends | Use paired lake-area analysis where possible; propose richer time-series analysis when more dates are collected |
| **Multimodal learning** | Fuses optical imagery, SAR, glacier velocity, land-surface temperature, precipitation and topography | Modalities have different resolutions, missingness patterns and time coverage; strong leakage risks | Design recommendation: harmonised lake-time features plus quality indicators and calibrated outputs |
| **Physically based breach / flood routing (e.g. HEC-RAS or other hydrodynamic models)** | Estimates flood hydrograph, runout, depth, arrival time and exposed infrastructure under scenarios | Requires bathymetry, breach assumptions, DEMs and channel/friction data; scenario uncertainty can be large | Treat this as the consequence layer, not a replacement for hazard monitoring |

### Selected literature and concrete findings

- **Rashid & Kashani, Journal of Glaciology (published online 2 January 2026):** a Kashmir Himalaya study manually mapped 155 lakes from satellite imagery for 1992 and 2024, used ten triggering factors with an Analytic Hierarchy Process (AHP), and reported five very-high, 35 high, 77 moderate, 32 low and six very-low susceptibility lakes. The AHP consistency ratio was reported as 0.07. Eight previously outburst-affected lakes used for retrospective checking were all placed in the very-high category. This is a useful independent benchmark, but eight selected positives and no matched negatives do **not** establish specificity, precision, calibration or prospective forecasting skill.
- **Qu et al., *Journal of Hydrology: Regional Studies* (2025), South Lhonak:** Sentinel-1 time series and a U-Net were used to map the lake from 2016–2023; the paper reports expansion from about 1.12 km² to 1.63 km² (about 45.5%) and identifies an ice/rock avalanche as the immediate trigger in its synthesis.
- **ISRO / NRSC, 4 October 2023:** the satellite-based rapid assessment compared RISAT-1A and Sentinel-1 observations and estimated that about **105 hectares** had drained between the 28 September and 4 October views. This is a published reference finding, not a value computed by this notebook.
- **Shrestha et al., *Earth System Science Data* (2023), HMA GLOF database:** introduced a version-controlled High Mountain Asia event inventory (697 events in the 2023 paper). The ICIMOD database page reports a December 2025 update documenting 766 events for 1533–2025. The counts differ because database versions and curation dates differ; always record the version used.
- **Nurakynov et al., *Geosciences* (2025), remote-sensing review:** reviews remote sensing for GLOF monitoring and risk assessment.
- **“Monitoring Earth’s glacial lakes from space with machine learning,” *Science of Remote Sensing* (December 2025):** reviews learning-based lake monitoring, notes the prominence of U-Net/DeepLab-type approaches, and stresses transferability and fair benchmarking.
- **GLOFNet multimodal dataset paper (conference proceedings indexed in 2026):** proposes combining Sentinel-2, NASA ITS_LIVE glacier velocity and MODIS land-surface temperature for a Shisper Glacier case study. It is a useful direction, but should be benchmarked with geographically and temporally held-out data before treating its outputs as actionable forecasts.

**Selected source links** (also listed at the end):
- Kashmir lake dataset: https://zenodo.org/records/15051825
- Kashmir susceptibility paper: https://www.cambridge.org/core/journals/journal-of-glaciology/article/glacial-lake-outburst-flood-susceptibility-and-potential-downstream-implications-across-the-kashmir-himalaya/501837D9ABC0A452DD68841F1225D78D
- ISRO / NRSC South Lhonak satellite study: https://new1.isro.gov.in/ISRO_EN/Satellite_studies_South_Lhonak_Lake.html
- South Lhonak event paper: https://doi.org/10.1016/j.ejrh.2025.102578
- HMA event inventory repository: https://github.com/fidelsteiner/HMAGLOFDB
- HMA database paper: https://doi.org/10.5194/essd-15-3941-2023
- 2025 remote-sensing review: https://doi.org/10.3390/geosciences15060211
- Machine-learning review: https://doi.org/10.1016/j.srs.2025.100277
- GLOFNet: https://doi.org/10.1109/ICoDT269104.2025.11360730

---

## 3. Data strategy and reproducibility

### Primary lake-level dataset

The Zenodo record *Glacial Lake Outburst Flood susceptibility and potential downstream implications across the Kashmir Himalaya* supplies `GLOF_susceptibility` and `Lake_dynamics` shapefiles. Its description states that it contains lake dynamics for 1992–2024 and ten GLOF-triggering factors. The associated paper describes the underlying inventory as 155 lakes, mapped at least in the 1992 and 2024 snapshots. These are real geospatial records derived from remote sensing and GIS interpretation.

### Event inventory

The notebook attempts to retrieve the maintained High Mountain Asia GLOF database from the authors’ public GitHub repository. If this fails (network restrictions, repository layout changes, or service rate limits), the rest of the analysis still runs. Event counts are descriptive because reporting completeness varies by place and time.

### Satellite imagery

The optional South Lhonak module uses the Microsoft Planetary Computer STAC catalogue to search the public Sentinel-2 Level-2A archive around 26 September and 6 October 2023. It chooses the nearest candidate scenes in the search window and computes an exploratory NDWI water-like mask after masking cloud/shadow/snow classes from SCL. This quick analysis is **not** a substitute for expert lake delineation or an operational alert. Imagery availability, tile choice and mask quality are reported by the code.

### Limitations known before fitting

1. The provided susceptibility labels are based on an expert-weighted risk index. The ML comparison evaluates agreement with those labels, not real event outcomes.
2. There are only 155 lakes in the primary inventory; a deep neural network trained from scratch would be inappropriate here.
3. The paper’s retrospective check of eight historical outburst lakes is a small, positive-only benchmark; it cannot estimate false-alarm rate.
4. In the satellite case study, two scenes do not define a robust trend; there are no independently hand-labelled masks in this notebook.
5. The dataset and library versions are recorded at runtime. Results should not be quoted until the notebook has completed data download and execution.

## 4. Environment setup

Run this notebook in an internet-enabled Jupyter environment (or Google Colab). It downloads openly available data from Zenodo/GitHub and, in the optional section, cloud-hosted Sentinel-2 data. If installing in a managed environment, use a virtual environment and check institutional policy first.

In [ ]:
# If needed, uncomment and run once. The geospatial packages can take a few minutes to install.
# %pip install -q numpy pandas scipy matplotlib scikit-learn geopandas shapely pyproj requests odfpy
# Optional satellite section:
# %pip install -q pystac-client planetary-computer rasterio

In [ ]:
from __future__ import annotations
import os, re, json, math, sys, time, warnings, platform
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (balanced_accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
from sklearn.inspection import permutation_importance
import requests

try:
    import geopandas as gpd
    from shapely.geometry import box, mapping
    from shapely.ops import transform as shapely_transform
except Exception as exc:
    raise ImportError("Install geopandas and shapely using the setup cell before continuing.") from exc

SEED = 2026
np.random.seed(SEED)
PROJECT_DIR = Path.cwd() / "glof_project"
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__, "| geopandas:", gpd.__version__)
print("Project directory:", PROJECT_DIR.resolve())

## 5. Download and load the real lake inventory

The downloader uses the Zenodo API rather than a fragile hand-copied file URL. It downloads only the shapefile layers and their sidecar files. If an API link changes, the error message prints the URL and allows manual retrieval from the source record.

In [ ]:
ZENODO_RECORD_ID = "15051825"

def download_zenodo_shapefiles(record_id: str, destination: Path) -> list[Path]:
    """Download shapefile parts for the GLOF susceptibility and lake dynamics layers."""
    destination.mkdir(parents=True, exist_ok=True)
    api_url = f"https://zenodo.org/api/records/{record_id}"
    response = requests.get(api_url, timeout=60)
    response.raise_for_status()
    metadata = response.json()
    files = metadata.get("files", [])
    if not files:
        raise RuntimeError(f"No files returned by Zenodo API: {api_url}")

    stems = ("GLOF_susceptibility", "Lake_dynamics")
    useful_ext = {".shp", ".shx", ".dbf", ".prj", ".cpg", ".sbn", ".sbx", ".qix"}
    downloaded = []
    for item in files:
        key = item.get("key", "")
        if not any(Path(key).stem.lower().startswith(stem.lower()) for stem in stems):
            continue
        if Path(key).suffix.lower() not in useful_ext:
            continue
        target = destination / Path(key).name
        if target.exists() and target.stat().st_size > 0:
            downloaded.append(target)
            continue
        links = item.get("links", {})
        urls = [links.get("self"), links.get("content"),
                f"https://zenodo.org/records/{record_id}/files/{quote(key)}?download=1"]
        last_error = None
        for url in [u for u in urls if u]:
            try:
                with requests.get(url, stream=True, timeout=120) as r:
                    r.raise_for_status()
                    temp = target.with_suffix(target.suffix + ".part")
                    with open(temp, "wb") as f:
                        for chunk in r.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)
                    temp.replace(target)
                if target.stat().st_size == 0:
                    target.unlink(missing_ok=True)
                    raise IOError("Downloaded an empty file")
                downloaded.append(target)
                break
            except Exception as exc:
                last_error = exc
        else:
            raise RuntimeError(f"Could not download {key}. Source: {api_url}. Last error: {last_error}")

    shp_files = list(destination.glob("*.shp"))
    if not shp_files:
        raise RuntimeError("No .shp files downloaded. Open https://zenodo.org/records/15051825 and download the shapefile components manually.")
    return shp_files

shp_paths = download_zenodo_shapefiles(ZENODO_RECORD_ID, DATA_DIR)
print("Shapefiles:")
for p in sorted(shp_paths):
    print(f"  {p.name:35s} {p.stat().st_size/1024:8.1f} KB")

In [ ]:
sus_candidates = [p for p in shp_paths if "susceptibility" in p.stem.lower()]
dyn_candidates = [p for p in shp_paths if "lake_dynamics" in p.stem.lower()]
if not sus_candidates:
    raise FileNotFoundError("The susceptibility shapefile was not found in the downloaded record.")

lakes = gpd.read_file(sus_candidates[0])
dynamics = gpd.read_file(dyn_candidates[0]) if dyn_candidates else None

print("Susceptibility layer shape:", lakes.shape)
print("CRS:", lakes.crs)
print("Geometry types:", lakes.geometry.geom_type.value_counts(dropna=False).to_dict())
print("Columns:", lakes.columns.tolist())
display(lakes.drop(columns="geometry", errors="ignore").head())

if dynamics is not None:
    print("\nLake-dynamics layer shape:", dynamics.shape)
    print("Columns:", dynamics.columns.tolist())
    display(dynamics.drop(columns="geometry", errors="ignore").head())

## 6. Data-quality and spatial exploratory analysis

The first checks are not decorative: geometry validity, missing attributes, unexpected categories and spatial coverage can be sources of silent modelling errors. The map below uses the class field only when a categorical susceptibility field is present; otherwise it maps the full inventory without pretending a class exists.

In [ ]:
# General data-quality report
qa = pd.DataFrame({
    "dtype": lakes.drop(columns="geometry", errors="ignore").dtypes.astype(str),
    "missing_n": lakes.drop(columns="geometry", errors="ignore").isna().sum(),
    "missing_pct": (100*lakes.drop(columns="geometry", errors="ignore").isna().mean()).round(1),
    "n_unique": lakes.drop(columns="geometry", errors="ignore").nunique(dropna=True),
})
display(qa.sort_values("missing_pct", ascending=False))
print("Invalid geometries:", int((~lakes.geometry.is_valid).sum()) if lakes.geometry.notna().any() else "geometry missing")
print("Empty geometries:", int(lakes.geometry.is_empty.sum()) if lakes.geometry.notna().any() else "geometry missing")

# Identify a categorical susceptibility label if present (exclude continuous scores/indexes).
col_map = {c: re.sub(r"[^a-z0-9]+", "", str(c).lower()) for c in lakes.columns if c != "geometry"}
class_candidates = []
score_candidates = []
for c, key in col_map.items():
    nun = lakes[c].nunique(dropna=True)
    if any(t in key for t in ["gsi", "susceptibilityindex", "susceptibilityscore", "riskindex", "riskscore"]) or ("susceptib" in key and nun > 5):
        if pd.api.types.is_numeric_dtype(lakes[c]) and nun > 5:
            score_candidates.append(c)
    if any(t in key for t in ["susceptibility", "susceptib", "susclass", "suscat", "glofclass", "riskclass", "riskcategory", "hazardclass"]):
        if not any(t in key for t in ["index", "score", "gsi", "rank"]):
            if nun <= 10:
                class_candidates.append(c)
TARGET_COL = class_candidates[0] if class_candidates else None
SCORE_COL = score_candidates[0] if score_candidates else None

if TARGET_COL is None and SCORE_COL is not None:
    # Fallback: form 5 equal-frequency groups only for a descriptive susceptibility proxy.
    # This is not a new physical ground truth; document it as a score-derived label.
    TARGET_COL = "_derived_susceptibility_class"
    score = pd.to_numeric(lakes[SCORE_COL], errors="coerce")
    valid = score.notna()
    labels = ["Very low", "Low", "Moderate", "High", "Very high"]
    lakes[TARGET_COL] = pd.Series(index=lakes.index, dtype="object")
    try:
        lakes.loc[valid, TARGET_COL] = pd.qcut(score[valid], q=5, labels=labels, duplicates="drop").astype(str)
    except Exception:
        lakes.loc[valid, TARGET_COL] = pd.cut(score[valid], bins=5, labels=False, duplicates="drop").astype("Int64").astype(str)
    print(f"No class field found; created quintile labels from {SCORE_COL}. These labels are score-derived, not independent outcomes.")

print("Detected susceptibility class field:", TARGET_COL)
print("Detected continuous score field:", SCORE_COL)
if TARGET_COL is not None:
    print("Class counts:")
    display(lakes[TARGET_COL].value_counts(dropna=False).rename_axis("class").to_frame("count"))
    fig, ax = plt.subplots(figsize=(8, 4))
    lakes[TARGET_COL].fillna("Missing").value_counts().plot(kind="bar", ax=ax)
    ax.set_title("Distribution of susceptibility labels in the supplied inventory")
    ax.set_xlabel("Published / derived susceptibility class")
    ax.set_ylabel("Number of lakes")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(9, 8))
try:
    if TARGET_COL is not None:
        lakes.plot(column=TARGET_COL, categorical=True, legend=True, ax=ax, alpha=0.85, edgecolor="black", linewidth=0.25)
        ax.set_title("Spatial distribution of lakes by susceptibility class")
    else:
        lakes.plot(ax=ax, color="steelblue", edgecolor="black", linewidth=0.25)
        ax.set_title("Mapped glacial lake inventory")
    ax.set_xlabel("Longitude / projected x"); ax.set_ylabel("Latitude / projected y")
    ax.set_aspect("equal", adjustable="datalim")
    plt.tight_layout(); plt.show()
except Exception as exc:
    print("Map could not be rendered (often a geometry/CRS issue):", repr(exc))

### Published AHP weights and source-score audit

The recent Kashmir study uses AHP to weight ten factors. The published weights below sum to 0.995 because the displayed values are rounded; the second column normalises the rounded values to sum to one. This table is a **literature benchmark**, not a new estimate of expert preferences. Where the source file provides a continuous susceptibility index, the next cell audits its distribution and ranks records. It does not reinterpret the index as an event probability.

In [ ]:
AHP_WEIGHTS_2026 = pd.DataFrame([
    ("Steepest slope", 0.234), ("Dam material", 0.203), ("Upstream lake cascade", 0.165),
    ("Lake type / glacier contact", 0.106), ("Lake expansion rate", 0.091),
    ("Lake length-to-width ratio", 0.055), ("Seismic activity", 0.049), ("Lake area", 0.043),
    ("Dam-front slope gradient", 0.033), ("Permafrost occurrence", 0.016),
], columns=["factor", "published_rounded_weight"])
AHP_WEIGHTS_2026["normalised_rounded_weight"] = AHP_WEIGHTS_2026.published_rounded_weight / AHP_WEIGHTS_2026.published_rounded_weight.sum()
print("Sum of published rounded weights:", AHP_WEIGHTS_2026.published_rounded_weight.sum())
display(AHP_WEIGHTS_2026.sort_values("normalised_rounded_weight", ascending=False).round(4))

if SCORE_COL is not None and SCORE_COL in lakes.columns:
    score_values = pd.to_numeric(lakes[SCORE_COL], errors="coerce")
    print(f"\nSource susceptibility score field: {SCORE_COL}")
    display(score_values.describe().to_frame("source_score"))
    fig, ax = plt.subplots(figsize=(8, 4))
    score_values.dropna().hist(bins=18, ax=ax, edgecolor="white")
    ax.set_title("Distribution of source susceptibility index")
    ax.set_xlabel("Source index units (not probability)"); ax.set_ylabel("Lake count")
    plt.tight_layout(); plt.show()
    name_col = next((c for c in lakes.columns if c != "geometry" and any(t in re.sub(r"[^a-z0-9]+", "", str(c).lower()) for t in ["lakename", "name"])), None)
    show_cols = [c for c in [name_col, SCORE_COL, TARGET_COL] if c is not None and c in lakes.columns]
    ranking = lakes.dropna(subset=[SCORE_COL]).sort_values(SCORE_COL, ascending=False)[show_cols].head(10)
    print("Top 10 rows by source index (rank only; not a calibrated probability):")
    display(ranking)
    ranking.to_csv(OUTPUT_DIR / "top10_source_susceptibility_index.csv", index=False)
else:
    print("No continuous source index found; showing the published weights only. Do not infer a lake ranking from this table alone.")

## 7. Spatiotemporal analysis of lake dynamics

The source inventory documents snapshots in 1992 and 2024, not a continuous annual record for each lake. Therefore, the analysis below focuses on **paired change** where usable area columns exist. It reports a paired non-parametric test and a bootstrap interval where possible, but does not manufacture an annual trend from two observations. Units are taken from the source schema and should be verified against the data dictionary before external reporting.

In [ ]:
def normal_key(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())

def find_year_area_column(frame, year):
    if frame is None:
        return None
    candidates = []
    for c in frame.columns:
        if c == "geometry":
            continue
        k = normal_key(c)
        if str(year) in k and any(tok in k for tok in ["area", "ha", "km2", "shapearea", "lakeextent"]):
            candidates.append(c)
    return candidates[0] if candidates else None

area_frame = dynamics if dynamics is not None else lakes
AREA_1992_COL = find_year_area_column(area_frame, 1992)
AREA_2024_COL = find_year_area_column(area_frame, 2024)
print("Area source layer:", "Lake_dynamics" if dynamics is not None else "GLOF_susceptibility")
print("1992 area column:", AREA_1992_COL, "| 2024 area column:", AREA_2024_COL)

area_results = {}
if AREA_1992_COL and AREA_2024_COL:
    a = pd.to_numeric(area_frame[AREA_1992_COL], errors="coerce")
    b = pd.to_numeric(area_frame[AREA_2024_COL], errors="coerce")
    paired = pd.DataFrame({"area_1992": a, "area_2024": b}).dropna()
    paired = paired[(paired.area_1992 > 0) & (paired.area_2024 > 0)].copy()
    paired["absolute_change"] = paired.area_2024 - paired.area_1992
    paired["percent_change"] = 100 * paired.absolute_change / paired.area_1992
    print("Valid paired lakes:", len(paired))
    display(paired.describe().T)
    if len(paired) >= 2:
        # Paired Wilcoxon signed-rank test; handle identical values safely.
        try:
            w_stat, w_p = stats.wilcoxon(paired.area_2024, paired.area_1992, zero_method="wilcox", alternative="two-sided")
        except ValueError:
            w_stat, w_p = np.nan, np.nan
        # Non-parametric bootstrap interval for the median paired percentage change.
        rng = np.random.default_rng(SEED)
        vals = paired.percent_change.to_numpy()
        boot = np.array([np.median(rng.choice(vals, size=len(vals), replace=True)) for _ in range(4000)])
        lo, hi = np.quantile(boot, [0.025, 0.975])
        area_results = {
            "n_paired": int(len(paired)),
            "median_percent_change": float(np.median(vals)),
            "mean_percent_change": float(np.mean(vals)),
            "median_percent_change_bootstrap_ci95": [float(lo), float(hi)],
            "wilcoxon_p": float(w_p) if pd.notna(w_p) else None,
            "area_units": "as encoded in the source dataset; verify before reporting"
        }
        print("Median percentage area change:", round(np.median(vals), 2), "%")
        print("Bootstrap 95% CI for median percentage change:", (round(lo, 2), round(hi, 2)), "%")
        print("Wilcoxon paired test p-value:", w_p)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        axes[0].scatter(paired.area_1992, paired.area_2024, alpha=0.75)
        mn = float(min(paired.area_1992.min(), paired.area_2024.min()))
        mx = float(max(paired.area_1992.max(), paired.area_2024.max()))
        axes[0].plot([mn, mx], [mn, mx], "--", color="black", linewidth=1)
        axes[0].set_xlabel("Area in 1992 (source units)"); axes[0].set_ylabel("Area in 2024 (source units)")
        axes[0].set_title("Paired lake areas")
        axes[1].hist(vals, bins=18, edgecolor="white")
        axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
        axes[1].set_xlabel("Percentage change, 1992–2024"); axes[1].set_ylabel("Lake count")
        axes[1].set_title("Distribution of paired percentage change")
        plt.tight_layout(); plt.show()
        paired.to_csv(OUTPUT_DIR / "lake_area_change.csv", index=False)
    else:
        print("Too few valid pairs for inference.")
else:
    print("Could not safely identify both area fields from column names.")
    print("Available columns:", list(area_frame.columns))
    print("Inspect the source schema and update find_year_area_column() rather than guessing units or values.")

## 8. Spatially grouped ML benchmark (susceptibility-label emulation)

### Model design

The target is the supplied susceptibility class. For a transparent operational screen, the primary binary task groups **High + Very high** against all remaining classes, when the label vocabulary supports that interpretation. The comparators are:

- **Dummy baseline:** most-frequent class.
- **Logistic Regression:** a simple, interpretable linear baseline with one-hot categorical features and scaled numeric features.
- **Random Forest:** a non-linear tree ensemble that can represent interactions.

The validation groups are spatial clusters of lake centroids, so a test fold is geographically separated from its training folds more than in a random row split. This is still only a small-region, retrospective label-emulation experiment. Model probabilities must **not** be interpreted as calibrated real-world GLOF probabilities.

In [ ]:
def identify_feature_columns(frame, target_col, score_col=None):
    exclude_terms = ["id", "name", "lakecode", "code", "uuid", "latitude", "longitude", "lat", "lon",
                     "coordinate", "geometry", "gsi", "susceptibility", "susceptib", "riskclass", "riskcategory",
                     "riskindex", "riskscore", "hazardclass", "hazardindex", "score", "rank", "target", "label"]
    feats = []
    for c in frame.columns:
        if c == "geometry" or c == target_col or c == score_col:
            continue
        key = normal_key(c)
        if any(t in key for t in exclude_terms):
            continue
        if frame[c].nunique(dropna=True) <= 1:
            continue
        # A date field or a free-text notes field is not a useful predictor here.
        if pd.api.types.is_datetime64_any_dtype(frame[c]):
            continue
        if frame[c].dtype == object and frame[c].nunique(dropna=True) > max(80, 0.8*len(frame)):
            continue
        feats.append(c)
    return feats

ml_results = {}
permutation_summary = pd.DataFrame()
if TARGET_COL is None:
    print("No target class/score was identified. ML benchmark skipped rather than inventing labels.")
else:
    ml_frame = lakes.copy()
    if TARGET_COL not in ml_frame:
        raise KeyError(TARGET_COL)
    # Ensure valid target and drop empty geometries only for spatial grouping.
    ml_frame = ml_frame[ml_frame[TARGET_COL].notna()].copy()
    y_text = ml_frame[TARGET_COL].astype(str).str.lower().str.replace(r"[^a-z0-9]+", " ", regex=True).str.strip()
    has_named_high = y_text.str.contains(r"\bhigh\b", regex=True).any()
    has_named_low = y_text.str.contains(r"\blow\b", regex=True).any()
    if has_named_high and has_named_low:
        y = y_text.str.contains(r"\bhigh\b", regex=True).astype(int)
        task_name = "High/Very-high susceptibility versus other classes"
        class_names = ["Other classes", "High / Very high"]
    else:
        # If categories are ordered labels but not the standard English vocabulary, preserve a multiclass target.
        y, class_names = pd.factorize(y_text, sort=True)
        y = pd.Series(y, index=ml_frame.index)
        task_name = "Susceptibility-category reproduction (multiclass)"
        class_names = list(pd.Index(y_text).unique())

    feature_cols = identify_feature_columns(ml_frame, TARGET_COL, SCORE_COL)
    # Keep only predictors with enough observed values; imputation handles the rest.
    feature_cols = [c for c in feature_cols if ml_frame[c].notna().sum() >= max(10, int(0.1*len(ml_frame)))]
    X = ml_frame[feature_cols].copy()
    numeric_features = X.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features]
    print("Task:", task_name)
    print("Observations:", len(ml_frame), "| positive class count:", int(np.sum(y)) if getattr(y, "nunique", lambda: 2)() <= 2 else "multiclass")
    print("Predictors:", feature_cols)
    print("Numeric:", numeric_features)
    print("Categorical:", categorical_features)
    display(pd.DataFrame({"feature": feature_cols,
                          "dtype": [str(X[c].dtype) for c in feature_cols],
                          "missing_pct": [100*X[c].isna().mean() for c in feature_cols]}))

    # Build approximate spatial groups from projected lake centroids; if geometry is unusable, CV falls back to row groups.
    if "geometry" in ml_frame and ml_frame.geometry.notna().all():
        cent = ml_frame.to_crs("EPSG:32643").geometry.centroid
        coords = np.column_stack([cent.x.to_numpy(), cent.y.to_numpy()])
    else:
        coords = np.arange(len(ml_frame)).reshape(-1, 1).astype(float)
    from sklearn.cluster import KMeans
    n_groups = min(5, max(2, len(ml_frame)//20))
    groups = KMeans(n_clusters=n_groups, random_state=SEED, n_init=10).fit_predict(coords)
    unique_groups = np.unique(groups)
    n_splits = min(5, len(unique_groups))
    splitter = GroupKFold(n_splits=n_splits)

    def build_pipeline(model_name):
        numeric_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                                 ("scale", StandardScaler()) if model_name == "Logistic Regression" else ("passthrough", "passthrough")])
        categorical_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                                     ("encode", OneHotEncoder(handle_unknown="ignore"))])
        transformer = ColumnTransformer([
            ("numeric", numeric_pipe, numeric_features),
            ("categorical", categorical_pipe, categorical_features)
        ], remainder="drop")
        if model_name == "Logistic Regression":
            estimator = LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED)
        elif model_name == "Random Forest":
            estimator = RandomForestClassifier(n_estimators=400, min_samples_leaf=3,
                                               class_weight="balanced_subsample", random_state=SEED,
                                               n_jobs=-1)
        else:
            estimator = DummyClassifier(strategy="most_frequent")
        return Pipeline([("preprocess", transformer), ("model", estimator)])

    model_names = ["Dummy baseline", "Logistic Regression", "Random Forest"]
    oof_predictions = {name: np.full(len(X), -1, dtype=int) for name in model_names}
    oof_probabilities = {name: np.full(len(X), np.nan, dtype=float) for name in model_names}
    fold_records = []
    rf_perm = []
    y_array = np.asarray(y, dtype=int)

    for fold, (tr, te) in enumerate(splitter.split(X, y_array, groups), start=1):
        if len(np.unique(y_array[tr])) < 2:
            print(f"Skipping fold {fold}: only one class in training set.")
            continue
        for name in model_names:
            pipe = build_pipeline(name)
            pipe.fit(X.iloc[tr], y_array[tr])
            pred = pipe.predict(X.iloc[te]).astype(int)
            oof_predictions[name][te] = pred
            if hasattr(pipe, "predict_proba") and len(pipe.classes_) == 2:
                oof_probabilities[name][te] = pipe.predict_proba(X.iloc[te])[:, list(pipe.classes_).index(1)]
            fold_records.append({"fold": fold, "model": name,
                                 "balanced_accuracy": balanced_accuracy_score(y_array[te], pred),
                                 "macro_f1": f1_score(y_array[te], pred, average="macro", zero_division=0),
                                 "precision_macro": precision_score(y_array[te], pred, average="macro", zero_division=0),
                                 "recall_macro": recall_score(y_array[te], pred, average="macro", zero_division=0)})
            # Permutation importance on held-out spatial fold, only for RF, to avoid training-set importance inflation.
            if name == "Random Forest":
                try:
                    pim = permutation_importance(pipe, X.iloc[te], y_array[te], scoring="balanced_accuracy",
                                                 n_repeats=8, random_state=SEED+fold, n_jobs=-1)
                    rf_perm.append(pd.DataFrame({"feature": feature_cols,
                                                 "importance_mean": pim.importances_mean,
                                                 "importance_sd": pim.importances_std,
                                                 "fold": fold}))
                except Exception as exc:
                    print("Permutation importance skipped in fold", fold, ":", repr(exc))

    fold_metrics = pd.DataFrame(fold_records)
    if not fold_metrics.empty:
        metrics = fold_metrics.groupby("model").agg(
            balanced_accuracy_mean=("balanced_accuracy", "mean"),
            balanced_accuracy_sd=("balanced_accuracy", "std"),
            macro_f1_mean=("macro_f1", "mean"),
            macro_f1_sd=("macro_f1", "std"),
            macro_precision_mean=("precision_macro", "mean"),
            macro_recall_mean=("recall_macro", "mean"),
            folds_evaluated=("fold", "nunique")
        ).sort_values("balanced_accuracy_mean", ascending=False)
        display(metrics.round(3))
        metrics.to_csv(OUTPUT_DIR / "spatial_cv_model_comparison.csv")
        fold_metrics.to_csv(OUTPUT_DIR / "spatial_cv_fold_metrics.csv", index=False)

        best_name = metrics.index[0]
        valid = oof_predictions[best_name] >= 0
        y_eval = y_array[valid]
        pred_eval = oof_predictions[best_name][valid]
        print("Best model under mean spatial-fold balanced accuracy:", best_name)
        print("Out-of-fold classification report (susceptibility-label emulation only):")
        print(classification_report(y_eval, pred_eval, zero_division=0))
        cm = confusion_matrix(y_eval, pred_eval, labels=sorted(np.unique(y_array)))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(values_format="d", cmap="Blues")
        plt.title(f"Spatial out-of-fold confusion matrix — {best_name}")
        plt.tight_layout(); plt.show()
        ml_results = {
            "task": task_name,
            "n_observations": int(len(ml_frame)),
            "n_predictors": int(len(feature_cols)),
            "features": feature_cols,
            "n_spatial_groups": int(n_groups),
            "metrics": metrics.reset_index().to_dict(orient="records"),
            "best_model": best_name,
            "best_model_oof_balanced_accuracy": float(balanced_accuracy_score(y_eval, pred_eval)),
            "best_model_oof_macro_f1": float(f1_score(y_eval, pred_eval, average="macro", zero_division=0)),
            "label_semantics": "AHP/expert susceptibility class or class derived from susceptibility score, not observed future GLOF outcome"
        }
        if rf_perm:
            permutation_summary = (pd.concat(rf_perm).groupby("feature").agg(
                importance_mean=("importance_mean", "mean"),
                importance_sd_between_folds=("importance_mean", "std"),
                mean_within_fold_sd=("importance_sd", "mean")
            ).sort_values("importance_mean", ascending=False).reset_index())
            display(permutation_summary)
            permutation_summary.to_csv(OUTPUT_DIR / "rf_spatial_permutation_importance.csv", index=False)
            fig, ax = plt.subplots(figsize=(8, max(3.5, 0.35*len(permutation_summary.head(12)))))
            top = permutation_summary.head(12).sort_values("importance_mean")
            ax.barh(top.feature, top.importance_mean, xerr=top.importance_sd_between_folds.fillna(0))
            ax.axvline(0, color="black", linewidth=0.8)
            ax.set_xlabel("Decrease in balanced accuracy after permutation")
            ax.set_title("Held-out spatial-fold permutation importance (Random Forest)")
            plt.tight_layout(); plt.show()
    else:
        print("No valid CV folds; check class balance, missing values and spatial grouping.")

## 9. Historical event inventory: descriptive time and mechanism patterns

The High Mountain Asia event database is useful for contextualising the hazard, but reported event counts are affected by historical documentation, accessibility, publication intensity, and changes in detection. The code therefore reports inventory summaries and plots without treating the raw annual count as an unbiased measure of the physical event rate.

In [ ]:
def download_hma_glof_csv(destination: Path) -> Path | None:
    """Find and download the maintained HMA GLOF CSV using the public GitHub tree API."""
    repo = "fidelsteiner/HMAGLOFDB"
    # Pin the dataset snapshot so runs are reproducible; update only after reviewing the repository's data-quality note.
    data_ref = "7f339a8d86c2eacd9a87977e0fec32d271bdf07d"
    tree_url = f"https://api.github.com/repos/{repo}/git/trees/{data_ref}?recursive=1"
    try:
        r = requests.get(tree_url, timeout=40, headers={"Accept": "application/vnd.github+json"})
        r.raise_for_status()
        tree = r.json().get("tree", [])
        candidates = [x for x in tree if x.get("type") == "blob" and x.get("path", "").lower().endswith(".csv")
                      and "hmaglo fdb".replace(" ", "") in x.get("path", "").lower().replace("_", "").replace("-", "")]
        # Broader fallback: CSVs in the GLOF database directory, excluding removed/metadata files.
        if not candidates:
            candidates = [x for x in tree if x.get("type") == "blob" and x.get("path", "").lower().endswith(".csv")
                          and "glof" in x.get("path", "").lower()
                          and "removed" not in x.get("path", "").lower()]
        if not candidates:
            return None
        # Prefer the main inventory name, then shortest path.
        candidates.sort(key=lambda x: ("hmaglo fdb".replace(" ", "") not in x["path"].lower().replace("_", "").replace("-", ""), len(x["path"])))
        rel = candidates[0]["path"]
        raw_url = f"https://raw.githubusercontent.com/{repo}/{data_ref}/{rel}"
        data = requests.get(raw_url, timeout=90)
        data.raise_for_status()
        target = destination / Path(rel).name
        target.write_bytes(data.content)
        print("Pinned HMAGLOFDB commit:", data_ref)
        return target
    except Exception as exc:
        print("Current GitHub event inventory download failed:", repr(exc))
        return None

event_path = download_hma_glof_csv(DATA_DIR)
events = None
if event_path is not None:
    try:
        events = pd.read_csv(event_path, low_memory=False)
        print("Loaded event inventory:", event_path.name, "| rows:", len(events), "| columns:", events.columns.tolist())
        display(events.head())
    except Exception as exc:
        print("Could not parse downloaded event CSV:", repr(exc))
else:
    print("Event inventory unavailable in this session; lake-level analysis can still be completed.")
    print("Source: https://github.com/fidelsteiner/HMAGLOFDB")

# Flexible year extraction without assuming the source's exact schema.
# The repository documents quality control through 2025-12-31; later-dated records are excluded pending cross-check.
event_summary = {}
if events is not None and len(events):
    year_series = None
    chosen_date_col = None
    # The maintained inventory has approximate and exact year fields. Prefer a valid approximate year,
    # then fall back to a valid exact year on rows where the approximate year is unavailable.
    if "Year_approx" in events.columns or "Year_exact" in events.columns:
        approx = pd.to_numeric(events["Year_approx"], errors="coerce") if "Year_approx" in events.columns else pd.Series(np.nan, index=events.index)
        exact = pd.to_numeric(events["Year_exact"], errors="coerce") if "Year_exact" in events.columns else pd.Series(np.nan, index=events.index)
        approx = approx.where((approx >= 1500) & (approx <= 2025))
        exact = exact.where((exact >= 1500) & (exact <= 2025))
        year_series = approx.combine_first(exact)
        chosen_date_col = "Year_approx; fallback Year_exact; cutoff 2025-12-31"
    else:
        date_cols = [c for c in events.columns if any(t in normal_key(c) for t in ["year", "date", "time"])]
        for c in date_cols:
            s = events[c]
            numeric = pd.to_numeric(s, errors="coerce")
            numeric = numeric.where((numeric >= 1500) & (numeric <= 2025))
            if numeric.notna().sum() >= max(5, int(0.2*len(s))):
                year_series = numeric
                chosen_date_col = c
                break
            parsed = pd.to_datetime(s, errors="coerce", utc=True)
            yr = parsed.dt.year.astype(float)
            yr = yr.where((yr >= 1500) & (yr <= 2025))
            if yr.notna().sum() >= max(5, int(0.2*len(s))):
                year_series = yr
                chosen_date_col = c
                break
    print("Temporal field used:", chosen_date_col)
    if year_series is not None:
        event_years = year_series.dropna().astype(int)
        event_summary = {"inventory_rows": int(len(events)), "events_with_valid_year": int(len(event_years)),
                         "rows_without_usable_year_after_cutoff": int(len(events) - len(event_years)),
                         "min_year": int(event_years.min()) if len(event_years) else None,
                         "max_year": int(event_years.max()) if len(event_years) else None,
                         "year_column": chosen_date_col}
        print("Rows with usable year:", len(event_years), "| rows without usable year / outside cutoff:", len(events)-len(event_years))
        annual = event_years.value_counts().sort_index()
        annual_df = annual.rename("reported_events").rename_axis("year").reset_index()
        decade_df = annual_df.assign(decade=(annual_df.year//10)*10).groupby("decade", as_index=False).reported_events.sum()
        display(decade_df.tail(20))
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
        recent = annual_df[annual_df.year >= max(1900, int(annual_df.year.min()))]
        axes[0].plot(recent.year, recent.reported_events, marker=".", linewidth=1)
        axes[0].set_title("Reported events by year (inventory records)")
        axes[0].set_xlabel("Year"); axes[0].set_ylabel("Count")
        axes[1].bar(decade_df.decade.astype(str), decade_df.reported_events)
        axes[1].set_title("Reported events by decade")
        axes[1].set_xlabel("Decade starting year"); axes[1].set_ylabel("Count")
        axes[1].tick_params(axis="x", rotation=55)
        plt.tight_layout(); plt.show()
        annual_df.to_csv(OUTPUT_DIR / "hma_glof_annual_counts.csv", index=False)
        decade_df.to_csv(OUTPUT_DIR / "hma_glof_decade_counts.csv", index=False)
    else:
        print("Could not identify a reliable event-year field; inspect columns before doing temporal analysis.")

## 10. Optional live Sentinel-2 case study: South Lhonak Lake

The observation dates are chosen to bracket the documented October 2023 outburst. The code searches the Microsoft Planetary Computer catalogue, selects scenes close to the reference dates, reads small cloud-hosted raster windows, computes NDWI = (Green − NIR)/(Green + NIR), and uses the Scene Classification Layer (SCL) to exclude cloud/shadow/snow classes. The output is a **water-like pixel estimate in a rectangular study window**, not a validated lake-area measurement. Topographic shadows, sediment-laden water and water-boundary choices remain important sources of error.

This section is intentionally wrapped in a `try` block: data catalogue/raster URLs or package versions can change, and it should not prevent the non-imagery parts of the project from running.

In [ ]:
RUN_SATELLITE_DEMO = True  # Set False to skip live satellite retrieval.
satellite_results = {}

if RUN_SATELLITE_DEMO:
    try:
        import rasterio
        from rasterio.features import geometry_window
        from rasterio.warp import transform_geom, reproject, Resampling
        import planetary_computer
        from pystac_client import Client

        # Approximate rectangular AOI surrounding South Lhonak Lake (WGS84); this is not a lake boundary.
        south_lhonak_bbox = [88.11, 27.90, 88.27, 27.99]
        catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
        search = catalog.search(collections=["sentinel-2-l2a"], bbox=south_lhonak_bbox,
                                datetime="2023-09-15/2023-10-20",
                                query={"eo:cloud_cover": {"lt": 90}}, max_items=80)
        items = list(search.items())
        if not items:
            raise RuntimeError("No Sentinel-2 L2A STAC items returned for AOI/date window.")
        print("Candidate Sentinel-2 scenes:", len(items))
        scene_table = pd.DataFrame([{
            "datetime": item.datetime,
            "cloud_cover_pct": item.properties.get("eo:cloud_cover"),
            "mgrs_tile": item.properties.get("s2:mgrs_tile"),
            "item_id": item.id
        } for item in items]).sort_values("datetime")
        display(scene_table.head(20))

        # Keep the most represented MGRS tile to reduce cross-tile comparisons.
        tiles = [item.properties.get("s2:mgrs_tile") for item in items if item.properties.get("s2:mgrs_tile")]
        if tiles:
            chosen_tile = pd.Series(tiles).value_counts().index[0]
            items = [item for item in items if item.properties.get("s2:mgrs_tile") == chosen_tile]
            print("Using MGRS tile:", chosen_tile)

        targets = [pd.Timestamp("2023-09-26", tz="UTC"), pd.Timestamp("2023-10-06", tz="UTC")]
        selected = []
        for target_date in targets:
            ranked = sorted(items, key=lambda it: (
                abs((pd.Timestamp(it.datetime).tz_convert("UTC") - target_date).days),
                float(it.properties.get("eo:cloud_cover", 100))
            ))
            candidate = next((it for it in ranked if abs((pd.Timestamp(it.datetime).tz_convert("UTC") - target_date).days) <= 12), None)
            if candidate is None:
                raise RuntimeError(f"No scene within 12 days of target {target_date.date()}")
            selected.append(candidate)
        if selected[0].id == selected[1].id:
            raise RuntimeError("Nearest-scene selection returned the same acquisition for both reference dates.")

        def crop_asset(item, asset_key, bbox):
            if asset_key not in item.assets:
                raise KeyError(f"Asset {asset_key} not found in STAC item {item.id}; assets={list(item.assets)}")
            with rasterio.open(item.assets[asset_key].href) as src:
                geom = transform_geom("EPSG:4326", src.crs, mapping(box(*bbox)))
                win = geometry_window(src, [geom], pad_x=0, pad_y=0)
                arr = src.read(1, window=win)
                return arr, src.window_transform(win), src.crs, src.nodata

        def process_scene(item, bbox):
            signed = planetary_computer.sign(item)
            # Read 10 m Green/NIR; read the scene classification layer for masks.
            green, tr, crs, green_nodata = crop_asset(signed, "B03", bbox)
            nir, tr2, crs2, nir_nodata = crop_asset(signed, "B08", bbox)
            if green.shape != nir.shape or crs != crs2:
                raise RuntimeError("Green and NIR grids do not match; regridding is needed for this scene.")
            green = green.astype("float32"); nir = nir.astype("float32")
            denominator = green + nir
            ndwi = np.divide(green - nir, denominator, out=np.full_like(green, np.nan), where=denominator != 0)
            valid = np.isfinite(ndwi)
            if green_nodata is not None: valid &= green != green_nodata
            if nir_nodata is not None: valid &= nir != nir_nodata

            if "SCL" in signed.assets:
                scl, scl_tr, scl_crs, scl_nodata = crop_asset(signed, "SCL", bbox)
                scl_on_green = np.zeros(green.shape, dtype=np.uint8)
                reproject(source=scl, destination=scl_on_green,
                          src_transform=scl_tr, src_crs=scl_crs,
                          dst_transform=tr, dst_crs=crs,
                          resampling=Resampling.nearest)
                # SCL 0=no data, 1=saturated, 3=cloud shadow, 8/9=cloud, 10=cirrus, 11=snow/ice.
                valid &= ~np.isin(scl_on_green, [0, 1, 3, 8, 9, 10, 11])
            water_like = valid & (ndwi > 0.10)
            pixel_area_m2 = abs(tr.a * tr.e)
            area_km2 = float(water_like.sum() * pixel_area_m2 / 1e6)
            return {"item_id": item.id, "datetime": pd.Timestamp(item.datetime).isoformat(),
                    "cloud_cover_pct": item.properties.get("eo:cloud_cover"), "ndwi": ndwi,
                    "valid": valid, "water_like": water_like, "area_km2_rectangular_aoi": area_km2,
                    "transform": tr, "crs": str(crs)}

        processed = [process_scene(it, south_lhonak_bbox) for it in selected]
        satellite_results = {
            "scenes": [{k: v for k, v in r.items() if k not in {"ndwi", "valid", "water_like", "transform"}} for r in processed],
            "note": "Exploratory NDWI mask in a rectangular AOI; not a validated lake polygon or official lake area."
        }
        fig, axes = plt.subplots(2, 2, figsize=(12, 9))
        for j, res in enumerate(processed):
            d = pd.Timestamp(res["datetime"]).strftime("%Y-%m-%d")
            axes[0, j].imshow(np.where(res["valid"], res["ndwi"], np.nan), vmin=-0.5, vmax=0.7, cmap="BrBG")
            axes[0, j].set_title(f"NDWI, {d}\nCloud cover: {res['cloud_cover_pct']}%")
            axes[0, j].axis("off")
            axes[1, j].imshow(res["water_like"], cmap="Blues", interpolation="nearest")
            axes[1, j].set_title(f"Exploratory water-like mask\n{res['area_km2_rectangular_aoi']:.3f} km² in rectangle")
            axes[1, j].axis("off")
        plt.suptitle("South Lhonak Lake: exploratory Sentinel-2 change view (not a validated delineation)", y=1.01)
        plt.tight_layout(); plt.show()
        scene_df = pd.DataFrame([{k: v for k, v in r.items() if k not in {"ndwi", "valid", "water_like", "transform"}} for r in processed])
        scene_df.to_csv(OUTPUT_DIR / "south_lhonak_sentinel2_exploratory.csv", index=False)
        display(scene_df)
        print("Reference for interpretation: ISRO/NRSC reported about 105 ha drained between 28 Sep and 4 Oct 2023.")
        print("Do not compare that reference directly with this rectangular-AOI mask as if they were the same estimator.")
    except Exception as exc:
        satellite_results = {"status": "skipped_or_failed", "reason": repr(exc)}
        print("Satellite demo did not complete in this environment:", repr(exc))
        print("The rest of the notebook is still usable. See ISRO's official event page:")
        print("https://new1.isro.gov.in/ISRO_EN/Satellite_studies_South_Lhonak_Lake.html")
else:
    satellite_results = {"status": "disabled_by_user"}
    print("Satellite demo disabled by RUN_SATELLITE_DEMO=False.")

## 11. Putting the layers together: an interpretable screening workflow

A defensible real-world architecture should keep three scores separate until the decision layer:

- **Susceptibility / hazard potential:** geomorphic stability, stored water, dam type and freeboard, upstream avalanche source areas, glacier/lake connectivity, lake growth and seismic setting.
- **Monitoring state / trigger evidence:** new lake-area or water-level change, glacier velocity and deformation anomalies, rainfall/temperature extremes, avalanche/landslide detections, SAR/optical disagreement and data quality.
- **Downstream exposure and consequence:** flow path and travel time, settlements, bridges, roads, hydropower infrastructure, evacuation access and uncertainty in breach scenarios.

The operational output should be a *review queue* for experts (priority tier, evidence, confidence, and next data needed), not an unreviewed automatic emergency alarm. A threshold is chosen with users according to false-alarm cost and missed-event cost, not from accuracy alone.

In [ ]:
# A simple reference architecture (conceptual, not a fitted probability model).
from IPython.display import display, Markdown
architecture = r"""
            OPEN DATA + FIELD OBSERVATIONS
      Sentinel-1/2 | DEM | climate | glacier velocity
                         |
                         v
             QC + lake-level data fusion
       cloud/snow checks, co-registration, metadata
                         |
          +--------------+---------------+
          |              |               |
          v              v               v
   Static hazard    Temporal change   Trigger context
   susceptibility   / anomaly model   (rain, heat, quake,
   (GIS/AHP/ML)     (stats/CV/TS)      avalanche, velocity)
          |              |               |
          +--------------+---------------+
                         v
        Expert-reviewed lake monitoring priority
                         |
                         v
        Breach scenarios + hydraulic routing model
                         |
                         v
    Downstream exposure + uncertainty + response plan
"""
display(Markdown("```text\n" + architecture + "\n```"))

## 12. Automated analysis report

This cell creates a Markdown report containing actual run-time summaries when data were successfully loaded. Do not quote the quantitative ML results from a run that failed to download data or skipped validation. External published results are clearly marked as literature references rather than notebook-generated measurements.

In [ ]:
from datetime import datetime

def safe_float(x, digits=3):
    try:
        return round(float(x), digits) if x is not None and np.isfinite(float(x)) else "NA"
    except Exception:
        return "NA"

report_lines = [
    "# GLOF susceptibility and monitoring prototype — run report",
    "",
    f"**Generated:** {datetime.now().astimezone().isoformat(timespec='seconds')}",
    "",
    "## Data loaded",
    f"- Kashmir susceptibility layer: {len(lakes)} rows; CRS `{lakes.crs}`.",
    f"- Susceptibility class field: `{TARGET_COL}`.",
    f"- Continuous score field: `{SCORE_COL}`.",
    f"- Area layer available: {'yes' if dynamics is not None else 'no'}.",
    f"- Historical event inventory raw rows: {event_summary.get('inventory_rows', 'not available')}.",
    f"- Event inventory rows with usable year: {event_summary.get('events_with_valid_year', 'not available')}.",
    f"- Event year range under the 2025 cutoff: {event_summary.get('min_year', 'NA')}–{event_summary.get('max_year', 'NA')}.",
    f"- Event inventory rows without a usable year under the 2025 cutoff: {event_summary.get('rows_without_usable_year_after_cutoff', 'not available')}.",
    "",
    "## Statistical lake-dynamics results",
]
if area_results:
    report_lines += [
        f"- Valid paired lake records: {area_results['n_paired']}.",
        f"- Median percentage area change: {safe_float(area_results['median_percent_change'], 2)}%.",
        f"- Mean percentage area change: {safe_float(area_results['mean_percent_change'], 2)}%.",
        f"- Bootstrap 95% interval for median percentage change: {area_results['median_percent_change_bootstrap_ci95']}%.",
        f"- Paired Wilcoxon test p-value: {safe_float(area_results['wilcoxon_p'], 5)}.",
        f"- Area units: {area_results['area_units']}.",
    ]
else:
    report_lines += ["Paired 1992/2024 area columns were not identified; no paired inference was run."]

report_lines += ["", "## Susceptibility ML validation", ""]
if ml_results:
    report_lines += [
        f"- Task: {ml_results['task']}.",
        f"- Observations / predictors: {ml_results['n_observations']} / {ml_results['n_predictors']}.",
        f"- Validation: grouped spatial cross-validation with {ml_results['n_spatial_groups']} geographic clusters.",
        f"- Best model by mean spatial-fold balanced accuracy: {ml_results['best_model']}.",
        f"- Best-model pooled out-of-fold balanced accuracy: {safe_float(ml_results['best_model_oof_balanced_accuracy'])}.",
        f"- Best-model pooled out-of-fold macro-F1: {safe_float(ml_results['best_model_oof_macro_f1'])}.",
        "- **Interpretation limit:** metrics measure reproduction of susceptibility classes / score-derived classes, not prediction of observed future GLOFs.",
    ]
    report_lines += ["", "### Spatial-fold model table", "", "| Model | Mean balanced accuracy | SD | Mean macro-F1 | Folds |", "|---|---:|---:|---:|---:|"]
    for row in ml_results["metrics"]:
        report_lines.append(f"| {row['model']} | {safe_float(row['balanced_accuracy_mean'])} | {safe_float(row.get('balanced_accuracy_sd'))} | {safe_float(row['macro_f1_mean'])} | {row['folds_evaluated']} |")
else:
    report_lines.append("No valid supervised model was fitted because no suitable target or validation folds were available.")

report_lines += ["", "## Satellite case study", ""]
if satellite_results.get("scenes"):
    report_lines.append("Sentinel-2 scenes were retrieved and processed. The scene table and masks were saved to the output directory. The computed water-like area is an exploratory rectangular-AOI estimate and **not** an authoritative lake outline.")
elif satellite_results.get("status") == "disabled_by_user":
    report_lines.append("The satellite demo was disabled by configuration.")
else:
    report_lines.append(f"Live Sentinel-2 processing did not complete: `{satellite_results.get('reason', 'not run')}`. Consult the official ISRO/NRSC study for the reference scene analysis.")

report_lines += [
    "", "## Literature reference findings (not computed by this notebook)",
    "- ISRO/NRSC reported approximately 105 ha drained from South Lhonak Lake between the 28 September and 4 October 2023 satellite views.",
    "- A 2025 South Lhonak study reported lake expansion from approximately 1.12 km² (2016) to 1.63 km² (2023), about 45.5%, using its own remote-sensing workflow.",
    "- The 2026 Kashmir Himalaya study reports 155 lakes, with 5 very-high / 35 high / 77 moderate / 32 low / 6 very-low susceptibility classes and AHP consistency ratio 0.07. These are published results from that study, not calculations made here.",
    "", "## Decision-relevant limitations", 
    "1. A susceptibility class is not an event probability or forecast date.",
    "2. Small sample, regional restriction, expert-label uncertainty and possible spatial dependence limit generalisation.",
    "3. Historical event inventories are incomplete and largely positive-only; the notebook does not claim independent event-prediction sensitivity/specificity.",
    "4. The optical mask is not manually validated; SCL and NDWI thresholds can misclassify shadow, snow/ice, sediment and narrow water channels.",
    "5. For a true forecasting study, construct a lake-year panel and evaluate held-out future periods and held-out regions using PR-AUC, recall at an operational alert budget, calibration/Brier score, event lead time, and false alarms per lake-year.",
    "", "## Outputs", 
    "The notebook writes reproducible tables and figures to `glof_project/outputs/` where the corresponding analysis completed.",
]
report_text = "\n".join(report_lines)
(OUTPUT_DIR / "analysis_report.md").write_text(report_text, encoding="utf-8")
print(report_text)
print("\nReport saved to:", OUTPUT_DIR / "analysis_report.md")

## 13. Conclusions, next experiments and interview defence

### What this project demonstrates

- You can operationalise a geospatial research question into a reproducible data pipeline.
- You understand that the statistical unit matters: lakes, images, events and lake-years should not be mixed indiscriminately.
- You know how to use robust metrics and spatial validation rather than accuracy alone.
- You can combine interpretable statistical models with remote sensing and physical process knowledge.
- You are explicit about epistemic limits: susceptibility labels are not the same as event outcomes; image classification is not flood forecasting.

### Next experiments worth proposing to the CVPR Unit

1. **Build the lake-time panel:** consistent lake IDs; pre-event and matched non-event observations; same seasonal windows; exact event dates and uncertainty.
2. **Extract aligned multimodal features:** optical reflectance / NDWI, Sentinel-1 SAR backscatter, DEM slope and dam-front geometry, distance/contact with glacier, upstream avalanche source potential, lake growth, glacier velocity, land-surface temperature, precipitation, seismic events, and missingness/cloud flags.
3. **Establish honest baselines:** logistic regression / GAM and a rule-based AHP baseline before Random Forest / gradient boosting; report PR-AUC, sensitivity at fixed false-alarm burden, Brier score/calibration, and decision curves.
4. **Use leakage-resistant evaluation:** leave-one-region-out and train-on-past/test-on-future evaluation; lakes from the same drainage system must not be split across folds if that creates spatial leakage.
5. **Apply segmentation models carefully:** U-Net or DeepLab for lake polygons only after producing a georeferenced training/validation mask set; test out-of-region transfer and boundary uncertainty.
6. **Connect to consequences:** simulate plausible breach hydrographs and downstream routing, then overlay buildings, roads, bridges, hydropower and evacuation assets.
7. **Communicate risk probabilistically:** include uncertainty bands, data-quality flags and an expert-in-the-loop alert workflow.

### A 60-second interview explanation

> “I designed a reproducible GLOF susceptibility and monitoring pipeline around open lake-level and event-inventory data. The first layer quantifies lake dynamics and geomorphic susceptibility; the second attempts multitemporal Sentinel-2 change extraction; and the third compares transparent Logistic Regression and Random Forest baselines using spatially grouped validation. A central design decision is to distinguish the published AHP susceptibility category from an observed outburst: good classification of an expert-derived label is not evidence of prospective prediction. For genuine forecasting, I would assemble a lake-year panel, add SAR, glacier velocity, temperature, precipitation and topographic features, and evaluate on future time windows and unseen regions using PR-AUC, calibration, recall at a fixed false-alarm budget and lead time. The consequence layer would then link hazard to downstream hydraulic routing and exposure.”

### Likely questions and concise answers

**Why not train a CNN immediately?**  The current lake-level dataset has only about 155 records, not thousands of annotated image masks. A CNN trained without suitable masks would produce an impressive-looking but scientifically weak result. First build a validated segmentation dataset, then benchmark U-Net/DeepLab against NDWI and traditional classifiers.

**Why spatial cross-validation?**  Nearby lakes share climate, terrain, drainage and glaciological conditions. Random splitting can place near-duplicates or correlated observations in train and test sets, inflating metrics. Spatial groups better approximate transfer to a new area.

**Why is accuracy inadequate?**  High-risk lake labels / actual events are a minority. A model can get high accuracy by predicting “not high-risk” almost everywhere. Balanced accuracy, macro-F1, precision-recall curves and recall under an alert budget reveal more; calibration is needed if probabilities guide decisions.

**Can you predict the exact date of a GLOF?**  Not from this dataset. A susceptibility inventory alone supports relative screening, not short-horizon time-to-event prediction. A time-to-event model needs repeatedly observed lake-years and dependable event/negative periods.

**What is the biggest scientific challenge?**  The hardest part is not choosing a complex model; it is constructing consistent labels and a validation design that represents prospective use, given rare events, incomplete inventories, cloud gaps, spatial dependence and changing observation quality.

**What makes the work multidisciplinary?**  It combines remote sensing and computer vision, spatial statistics, machine learning, glaciology/geomorphology, hydrology, uncertainty quantification and disaster-risk decision-making.

---

## References

1. Rashid, I. & Kashani, S. D. R. (online 2 Jan 2026). *Glacial lake outburst flood susceptibility and potential downstream implications across the Kashmir Himalaya.* Journal of Glaciology. https://www.cambridge.org/core/journals/journal-of-glaciology/article/glacial-lake-outburst-flood-susceptibility-and-potential-downstream-implications-across-the-kashmir-himalaya/501837D9ABC0A452DD68841F1225D78D
2. Rashid, I. & Kashani, S. D. R. (2025). *Glacial Lake Outburst Flood susceptibility and potential downstream implications across the Kashmir Himalaya* [dataset]. Zenodo. https://doi.org/10.5281/zenodo.15051825
3. Qu, G. et al. (2025). *Comprehensive analysis of the catastrophic October 4th 2023 glacial lake outburst flood in the South Lhonak Lake, Sikkim Himalayas.* Journal of Hydrology: Regional Studies, 60, 102578. https://doi.org/10.1016/j.ejrh.2025.102578
4. NRSC/ISRO (4 Oct 2023). *Satellite image-based studies on the outburst of South Lhonak Lake.* https://new1.isro.gov.in/ISRO_EN/Satellite_studies_South_Lhonak_Lake.html
5. Shrestha, F. et al. (2023). *A comprehensive and version-controlled database of glacial lake outburst floods in High Mountain Asia.* Earth System Science Data, 15, 3941–3961. https://doi.org/10.5194/essd-15-3941-2023
6. ICIMOD. *GLOF database of High Mountain Asia* (page notes update in December 2025). https://rds.icimod.org/metadata/8881454b-6f7c-461b-95c2-eaf7618230d9
7. Nurakynov, S. et al. (2025). *Advancements in Remote Sensing for Monitoring and Risk Assessment of Glacial Lake Outburst Floods.* Geosciences, 15(6), 211. https://doi.org/10.3390/geosciences15060211
8. *Monitoring Earth’s glacial lakes from space with machine learning.* Science of Remote Sensing (Dec 2025), 12, 100277. https://doi.org/10.1016/j.srs.2025.100277
9. *GLOFNet—A Multimodal Dataset for GLOF Monitoring and Prediction.* 2025 conference proceedings indexed in 2026. https://doi.org/10.1109/ICoDT269104.2025.11360730
10. Microsoft Planetary Computer STAC API. https://planetarycomputer.microsoft.com/docs/quickstarts/using-stac/
11. Copernicus Data Space Ecosystem STAC API documentation. https://documentation.dataspace.copernicus.eu/APIs/STAC.html

*This is a research prototype for discussion and reproducibility, not an operational warning system. Verify data licences, source versions, units, acquisition dates and model outputs before use in scientific or public-safety decisions.*